# MHCflurry CPU 예제
MHC class I 결합 친화도와 항원 처리 과정을 함께 고려해 펩타이드 제시를 예측한다.
## 실습 개요
- **활용 분야** — 면역원성 예측
- **카테고리** — Immunology
- **모델 유형** — Neural network (ensemble)
- **발표 연도** — 2020
- **라이선스** — Apache-2.0
**관련 논문**

- [MHCflurry 2.0: Improved Pan-Allele Prediction of MHC Class I-Presented Peptides by Incorporating Antigen Processing](https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30239-8) — Timothy J. O'Donnell 외, Cell Systems, 2020
- [MHCflurry: Open-Source Class I MHC Binding Affinity Prediction](https://www.cell.com/cell-systems/fulltext/S2405-4712(18)30232-1) — Timothy J. O'Donnell 외, Cell Systems, 2018

> **CPU 예제입니다.** 아래 예제는 한 개의 SMILES 또는 짧은 펩타이드 입력을 사용합니다. 플랫폼과 입력 방식·패키지 버전·산출물 종류가 다를 수 있습니다.

**노트북 패키지 버전:** 2.3.16. **플랫폼 카탈로그 버전:** 1.0.0. 서로 다른 버전의 수치를 동일한 결과로 비교하지 마세요. 설치된 버전은 결과 파일에 기록됩니다.

## 실행 조건

- **코드 저장소** — https://github.com/openvax/mhcflurry
- **가중치 확보** — 설치 후 `mhcflurry-downloads fetch` 로 사전학습 가중치를 내려받는다.
- **라이브러리** — 아래 환경 안내 참고
- **플랫폼에서 바로 실행** — https://lab.curieus.net/Analysis/mhcflurry

## 실행 제한사항 및 유의사항
- CPU 로 동작한다. GPU 런타임이 필요 없다.
- 가중치 내려받기에 수 분이 걸린다.
- 세션이 끊기면 설치한 패키지와 내려받은 가중치가 모두 사라진다. 재접속하면 설치부터 다시 해야 한다.
- 무료 런타임은 유휴 상태가 이어지면 연결이 끊긴다. 긴 실행은 중간 결과를 Google Drive 에 저장하는 편이 안전하다.

## 실행 절차

CPU 런타임에서 환경 준비 → 입력값 설정 → 실행 → 결과 확인 순서로 실행하세요. GPU 설정은 필요하지 않습니다.

이 예제는 짧은 펩타이드 두 개에 대해 MHC 결합·제시 관련 점수를 예측합니다. 플랫폼의 전체 FASTA 스캔 실습과는 입력 범위가 다릅니다.

## 환경 준비

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path
from sysconfig import get_path
SCRIPTS = Path(get_path("scripts"))
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # CPU example
os.environ['MPLBACKEND'] = 'Agg'  # CLI runs outside the notebook kernel
print('Python:', platform.python_version())
subprocess.run([sys.executable, "-m", "pip", "install", "mhcflurry==2.3.16"], check=True)

## 입력값 설정

In [ ]:
ALLELE = "HLA-A*02:01"  # @param {type:"string"}
PEPTIDES = ["SIINFEKL", "SYNFEKKL"]
subprocess.run([str(SCRIPTS / "mhcflurry-downloads"), "fetch", "models_class1_presentation"], check=True, timeout=1200)

## 실행

In [ ]:
from pathlib import Path
Path("preds.csv").unlink(missing_ok=True)
subprocess.run([str(SCRIPTS / "mhcflurry-predict"), "--alleles", ALLELE, "--peptides", *PEPTIDES, "--out", "preds.csv"], check=True, timeout=600)

## 결과 확인

In [ ]:
import json
import pandas as pd
from importlib.metadata import version
from IPython.display import display
result_path = Path("preds.csv")
if not result_path.is_file() or result_path.stat().st_size == 0:
    raise RuntimeError("Prediction output was not created")
predictions = pd.read_csv(result_path)
expected_rows = len(PEPTIDES)
if len(predictions) != expected_rows or predictions.select_dtypes(include="number").dropna(how="all").empty:
    raise RuntimeError("Prediction output is empty or incomplete")
required_scores = ["mhcflurry_affinity","mhcflurry_presentation_score"]
import numpy as np
if not set(required_scores).issubset(predictions.columns):
    raise RuntimeError("Expected model prediction columns are missing")
scores = predictions[required_scores].apply(pd.to_numeric, errors="raise")
if not np.isfinite(scores.to_numpy()).all():
    raise RuntimeError("Model prediction scores contain missing or non-finite values")
display(predictions)
environment = {"python": platform.python_version(), "package": "mhcflurry", "version": version("mhcflurry"), "platform_catalog_version": "1.0.0"}
Path("environment.json").write_text(json.dumps(environment, indent=2))
print(environment)
print("Saved: preds.csv, environment.json")

## 결과 해석

이 예제의 산출물은 펩타이드별 예측 CSV입니다. 결합 친화도는 낮을수록 강한 결합을 뜻하며, 제시 점수는 친화도와 다른 지표입니다. 실험적 면역원성을 입증하지 않습니다.

원본 논문과 저장소 문서의 지표 정의를 함께 확인하세요.